In [1]:
import duckdb
cn = duckdb.connect()
import os

sql = f"""
    CREATE OR REPLACE SECRET gcs_secrets (TYPE GCS, KEY_ID '{os.getenv("GCS_KEY")}', SECRET '{os.getenv("GCS_SECRET")}');
    
"""
cn.execute(sql)

In [2]:
cn.execute(f"ATTACH OR REPLACE 'ducklake:gcs_wh.ducklake' as gcs_wh (DATA_PATH 'gs://{os.getenv('GCS_BUCKET')}/ducklake');")

In [3]:
cn.sql("create table gcs_wh.orders (order_id integer, order_date date, customer_id integer, amount double)")
cn.sql("insert into gcs_wh.orders values (1, '2024-01-01', 1, 100.0), (2, '2024-01-02', 2, 150.0), (3, '2024-01-03', 1, 200.0)")
result = cn.sql("select * from gcs_wh.orders").fetchall()
print(result)

[(1, datetime.date(2024, 1, 1), 1, 100.0), (2, datetime.date(2024, 1, 2), 2, 150.0), (3, datetime.date(2024, 1, 3), 1, 200.0)]


In [4]:
cn.close()

In [5]:
# now open the ducklake db and examine it
cn = duckdb.connect('gcs_wh.ducklake')

In [8]:
cn.sql("select * from ducklake_table")

┌──────────┬──────────────────────────────────────┬────────────────┬──────────────┬───────────┬────────────┬─────────┬──────────────────┐
│ table_id │              table_uuid              │ begin_snapshot │ end_snapshot │ schema_id │ table_name │  path   │ path_is_relative │
│  int64   │                 uuid                 │     int64      │    int64     │   int64   │  varchar   │ varchar │     boolean      │
├──────────┼──────────────────────────────────────┼────────────────┼──────────────┼───────────┼────────────┼─────────┼──────────────────┤
│        1 │ 019c6e77-90de-77d1-9006-c980729cf7f8 │              1 │         NULL │         0 │ orders     │ orders/ │ true             │
└──────────┴──────────────────────────────────────┴────────────────┴──────────────┴───────────┴────────────┴─────────┴──────────────────┘

In [9]:
cn.sql("select * from ducklake_column")

┌───────────┬────────────────┬──────────────┬──────────┬──────────────┬─────────────┬─────────────┬─────────────────┬───────────────┬───────────────┬───────────────┐
│ column_id │ begin_snapshot │ end_snapshot │ table_id │ column_order │ column_name │ column_type │ initial_default │ default_value │ nulls_allowed │ parent_column │
│   int64   │     int64      │    int64     │  int64   │    int64     │   varchar   │   varchar   │     varchar     │    varchar    │    boolean    │     int64     │
├───────────┼────────────────┼──────────────┼──────────┼──────────────┼─────────────┼─────────────┼─────────────────┼───────────────┼───────────────┼───────────────┤
│         1 │              1 │         NULL │        1 │            1 │ order_id    │ int32       │ NULL            │ NULL          │ true          │          NULL │
│         2 │              1 │         NULL │        1 │            2 │ order_date  │ date        │ NULL            │ NULL          │ true          │          NULL │
│   